In [7]:
# Instalação das bibliotecas necessárias
!pip install pandas great_expectations -q

import pandas as pd
import io
from google.colab import files

# Solicita o upload do arquivo
print("Selecione o arquivo swiggy.csv original:")
uploaded = files.upload()

# Carrega o DataFrame
file_name = list(uploaded.keys())[0]
#df = pd.read_csv(io.BytesIO(uploaded[file_name]))
df = pd.read_excel(io.BytesIO(uploaded[file_name]))

print(f"\n✅ Arquivo '{file_name}' carregado com sucesso!")
print(f"Total de registros: {len(df)}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/5.0 MB 61.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.1/118.1 kB 8.6 MB/s eta 0:00:00
Selecione o arquivo swiggy.csv original:


Saving swiggy_data.xlsx to swiggy_data.xlsx

✅ Arquivo 'swiggy_data.xlsx' carregado com sucesso!
Total de registros: 197430


In [8]:
# 1. Ver as primeiras linhas e os nomes das colunas
print("--- Primeiras 5 linhas ---")
display(df.head())

# 2. Verificar tipos de dados e valores nulos
print("\n--- Informações do Dataset ---")
df.info()

# 3. Estatística Descritiva (O coração da sua análise)
print("\n--- Resumo Estatístico ---")
display(df.describe())

# 4. Verificar se há linhas duplicadas
print(f"\nLinhas duplicadas encontradas: {df.duplicated().sum()}")

--- Primeiras 5 linhas ---


,State,City,Order Date,Restaurant Name,Location,Category,Dish Name,Price (INR),Rating,Rating Count
0,Karnataka,Bengaluru,2025-06-29,Anand Sweets & Savouries,Rajarajeshwari Nagar,Snack,Butter Murukku-200gm,133.9,4.0,0
1,Karnataka,Bengaluru,2025-04-03,Srinidhi Sagar Deluxe,Kengeri,Recommended,Badam Milk,52.0,4.5,25
2,Karnataka,Bengaluru,2025-01-15,Srinidhi Sagar Deluxe,Kengeri,Recommended,Chow Chow Bath,117.0,4.7,48
3,Karnataka,Bengaluru,2025-04-17,Srinidhi Sagar Deluxe,Kengeri,Recommended,Kesari Bath,65.0,4.6,65
4,Karnataka,Bengaluru,2025-03-13,Srinidhi Sagar Deluxe,Kengeri,Recommended,Mix Raitha,130.0,4.0,0



--- Informações do Dataset ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 197430 entries, 0 to 197429
Data columns (total 10 columns):
 #   Column           Non-Null Count   Dtype         
---  ------           --------------   -----         
 0   State            197430 non-null  object        
 1   City             197430 non-null  object        
 2   Order Date       197430 non-null  datetime64[ns]
 3   Restaurant Name  197430 non-null  object        
 4   Location         197430 non-null  object        
 5   Category         197430 non-null  object        
 6   Dish Name        197430 non-null  object        
 7   Price (INR)      197430 non-null  float64       
 8   Rating           197430 non-null  float64       
 9   Rating Count     197430 non-null  int64         
dtypes: datetime64[ns](1), float64(2), int64(1), object(6)
memory usage: 15.1+ MB

--- Resumo Estatístico ---


,Order Date,Price (INR),Rating,Rating Count
count,197430,197430.000000,197430.000000,197430.000000
mean,2025-05-01 19:41:20.996808960,268.512920,4.341582,28.321805
min,2025-01-01 00:00:00,0.950000,1.500000,0.000000
25%,2025-03-01 00:00:00,139.000000,4.300000,0.000000
50%,2025-05-02 00:00:00,229.000000,4.400000,2.000000
75%,2025-07-01 00:00:00,329.000000,4.500000,15.000000
max,2025-08-31 00:00:00,8000.000000,5.000000,999.000000
std,NaN,219.338363,0.422585,87.542593



Linhas duplicadas encontradas: 27


limpeza & data quality
*texto en cursiva*

In [9]:
# 1. Limpeza de Duplicatas
df = df.drop_duplicates()
print(f"✅ Duplicatas removidas. Novo total: {len(df)}")

import great_expectations as ge
import pandas as pd

# 1. Configurar o contexto
context = ge.get_context()

# 2. Configurar o Datasource e o Batch Definition
datasource = context.data_sources.add_pandas(name="swiggy_datasource")
data_asset = datasource.add_dataframe_asset(name="swiggy_asset")
batch_definition = data_asset.add_batch_definition_whole_dataframe("all_data")

# 3. Criar a Suite de Expectativas
suite = context.suites.add(ge.ExpectationSuite(name="swiggy_report"))

# 4. Adicionar as regras (Expectations)
suite.add_expectation(ge.expectations.ExpectColumnValuesToBeBetween(
    column="Price (INR)", min_value=0.1, max_value=10000
))
suite.add_expectation(ge.expectations.ExpectColumnValuesToBeBetween(
    column="Rating", min_value=1.0, max_value=5.0
))
suite.add_expectation(ge.expectations.ExpectColumnValuesToNotBeNull(column="Restaurant Name"))
suite.add_expectation(ge.expectations.ExpectColumnValuesToNotBeNull(column="Order Date"))

# 5. Criar a Validação vinculando o Batch Definition
definition = ge.ValidationDefinition(
    name="swiggy_validation",
    data=batch_definition,
    suite=suite
)

# 6. Rodar a validação
result = definition.run(batch_parameters={"dataframe": df})

print("\n" + "="*40)
print("RELATÓRIO DE QUALIDADE (GREAT EXPECTATIONS)")
print("="*40)
print(f"Sucesso Geral: {result.success}")

stats = getattr(result, 'statistics', getattr(result, 'result_statistics', None))

if stats:
    # Se stats for um objeto Pydantic, convertemos para dict, se já for dict, usamos direto
    stats_dict = stats.dict() if hasattr(stats, 'dict') else stats

    print(f"Total de Testes: {stats_dict.get('evaluated_expectations')}")
    print(f"Sucessos: {stats_dict.get('successful_expectations')}")
    print(f"Percentual: {stats_dict.get('success_percent'):.2f}%")

    import json
    print("\n--- JSON  ---")
    print(json.dumps(stats_dict, indent=2))
else:
    print("\n--- RESULTADO DETALHADO ---")
    print(result)

✅ Duplicatas removidas. Novo total: 197403


INFO:great_expectations.data_context.types.base:Created temporary directory '/tmp/tmpfxllpwyn' for ephemeral docs site


Calculating Metrics:   0%|          | 0/27 [00:00<?, ?it/s]


RELATÓRIO DE QUALIDADE (GREAT EXPECTATIONS)
Sucesso Geral: True
Total de Testes: 4
Sucessos: 4
Percentual: 100.00%

--- JSON  ---
{
  "evaluated_expectations": 4,
  "successful_expectations": 4,
  "unsuccessful_expectations": 0,
  "success_percent": 100.0
}


In [10]:

    !pip install -q -U google-genai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.7/52.7 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 790.4/790.4 kB 19.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 246.5/246.5 kB 13.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.47.0, but you have google-auth 2.50.0 which is incompatible.


In [11]:
!pip install -q -U google-generativeai

# -q: Esta es una opción que significa "quiet" (silencioso). Reduce la cantidad de texto de salida que produce el comando, mostrando solo la información esencial.
# -U: Significa "upgrade" (actualizar). Esta opción le dice a pip que actualice el paquete especificado a la versión más reciente disponible.

In [12]:
# Importamos el paquete recien instalado
import google.generativeai as genai

# Esto es para usar secrets, la nueva opción de Google Colab, para almacenar la API key
from google.colab import userdata


All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)



In [13]:
import textwrap
from IPython.display import display
from IPython.display import Markdown

# Esta función se usa para dejar el formato Markdown que devuelve Gemini en formato compatible con Colab
def to_markdown(text):
  text = text.replace('•', '  *')
  return Markdown(textwrap.indent(text, '> ', predicate=lambda _: True))

### Añadiendo la API en Colab
En el panel lateral de Colab vas a ver una llave 🔑, dale click ahí. Luego "Add new secret" y en "Name" le vas a poner `GOOGLE_API_KEY` y en el "Value" vas a poner el texto que obtuviste en el sitio anterior.

In [14]:
# Configuramos nuestra instancia del modelo con nuestra API key
GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')

genai.configure(api_key = GEMINI_API_KEY)

In [15]:
# 2. Descobrir qual modelo está ativo para você
# Isso evita o erro 404 de digitar o nome errado
available_models = [m.name for m in genai.list_models() if 'generateContent' in m.supported_generation_methods]

print("Modelos disponíveis na sua conta:")
for name in available_models:
    print(f"- {name}")

# Selecionar o 1.5-flash se disponível, senão pega o primeiro da lista
target_model = next((m for m in available_models if '1.5-flash' in m), available_models[0])

print(f"\n🚀 Usando o modelo: {target_model}")
model = genai.GenerativeModel(target_model)

Modelos disponíveis na sua conta:
- models/gemini-2.5-flash
- models/gemini-2.5-pro
- models/gemini-2.0-flash
- models/gemini-2.0-flash-001
- models/gemini-2.0-flash-lite-001
- models/gemini-2.0-flash-lite
- models/gemini-2.5-flash-preview-tts
- models/gemini-2.5-pro-preview-tts
- models/gemma-3-1b-it
- models/gemma-3-4b-it
- models/gemma-3-12b-it
- models/gemma-3-27b-it
- models/gemma-3n-e4b-it
- models/gemma-3n-e2b-it
- models/gemma-4-26b-a4b-it
- models/gemma-4-31b-it
- models/gemini-flash-latest
- models/gemini-flash-lite-latest
- models/gemini-pro-latest
- models/gemini-2.5-flash-lite
- models/gemini-2.5-flash-image
- models/gemini-3-pro-preview
- models/gemini-3-flash-preview
- models/gemini-3.1-pro-preview
- models/gemini-3.1-pro-preview-customtools
- models/gemini-3.1-flash-lite-preview
- models/gemini-3-pro-image-preview
- models/nano-banana-pro-preview
- models/gemini-3.1-flash-image-preview
- models/lyria-3-clip-preview
- models/lyria-3-pro-preview
- models/gemini-3.1-flash-

In [16]:
 #3. Teste rápido com 2 pratos
test_dishes = ['Butter Murukku-200gm', 'Badam Milk']
prompt = f"Retorne um JSON com ingrediente principal e se é veg (true/false) para: {test_dishes}"

try:
    response = model.generate_content(prompt)
    print("\n--- RESPOSTA DA IA ---")
    print(response.text)
except Exception as e:
    print(f"Erro ao gerar conteúdo: {e}")


--- RESPOSTA DA IA ---
```json
[
  {
    "item": "Butter Murukku-200gm",
    "ingrediente_principal": "Manteiga",
    "veg": true
  },
  {
    "item": "Badam Milk",
    "ingrediente_principal": "Amêndoa",
    "veg": true
  }
]
```


In [17]:
#response = model.generate_content("¿Qué es la Felicidad?")

In [18]:
#print(response.text)

In [19]:
df

,State,City,Order Date,Restaurant Name,Location,Category,Dish Name,Price (INR),Rating,Rating Count
0,Karnataka,Bengaluru,2025-06-29,Anand Sweets & Savouries,Rajarajeshwari Nagar,Snack,Butter Murukku-200gm,133.9,4.0,0
1,Karnataka,Bengaluru,2025-04-03,Srinidhi Sagar Deluxe,Kengeri,Recommended,Badam Milk,52.0,4.5,25
2,Karnataka,Bengaluru,2025-01-15,Srinidhi Sagar Deluxe,Kengeri,Recommended,Chow Chow Bath,117.0,4.7,48
3,Karnataka,Bengaluru,2025-04-17,Srinidhi Sagar Deluxe,Kengeri,Recommended,Kesari Bath,65.0,4.6,65
4,Karnataka,Bengaluru,2025-03-13,Srinidhi Sagar Deluxe,Kengeri,Recommended,Mix Raitha,130.0,4.0,0
...,...,...,...,...,...,...,...,...,...,...
197425,Sikkim,Gangtok,2025-01-25,Mama's Kitchen,Gangtok,Momos,Soya cheese chilli momo ...,112.0,4.4,0
197426,Sikkim,Gangtok,2025-07-02,Mama's Kitchen,Gangtok,Momos,Kurkure momo fried ...,140.0,4.4,0
197427,Sikkim,Gangtok,2025-03-25,Mama's Kitchen,Gangtok,Momos,Chilli cheese momo,126.0,4.4,0
197428,Sikkim,Gangtok,2025-03-26,Mama's Kitchen,Gangtok,Momos,Veg Momos (8 Pc),85.0,4.4,0


###Abordagem

In [20]:
import re

def limpeza_express(texto):
    texto = str(texto).lower()
    # Remove tudo entre parênteses e colchetes (ex: (serves 1), [new])
    texto = re.sub(r'[\(\[].*?[\)\]]', '', texto)
    # Remove números e unidades (ex: 500ml, 2pcs, 10.50)
    texto = re.sub(r'\d+\s*(ml|gm|kg|pcs|pc|oz|g|l|ml)?', '', texto)
    # Remove símbolos estranhos
    texto = re.sub(r'[^a-zA-Z\s]', '', texto)
    return texto.strip()

df['dish_clean'] = df['Dish Name'].apply(limpeza_express)

In [21]:
df

,State,City,Order Date,Restaurant Name,Location,Category,Dish Name,Price (INR),Rating,Rating Count,dish_clean
0,Karnataka,Bengaluru,2025-06-29,Anand Sweets & Savouries,Rajarajeshwari Nagar,Snack,Butter Murukku-200gm,133.9,4.0,0,butter murukku
1,Karnataka,Bengaluru,2025-04-03,Srinidhi Sagar Deluxe,Kengeri,Recommended,Badam Milk,52.0,4.5,25,badam milk
2,Karnataka,Bengaluru,2025-01-15,Srinidhi Sagar Deluxe,Kengeri,Recommended,Chow Chow Bath,117.0,4.7,48,chow chow bath
3,Karnataka,Bengaluru,2025-04-17,Srinidhi Sagar Deluxe,Kengeri,Recommended,Kesari Bath,65.0,4.6,65,kesari bath
4,Karnataka,Bengaluru,2025-03-13,Srinidhi Sagar Deluxe,Kengeri,Recommended,Mix Raitha,130.0,4.0,0,mix raitha
...,...,...,...,...,...,...,...,...,...,...,...
197425,Sikkim,Gangtok,2025-01-25,Mama's Kitchen,Gangtok,Momos,Soya cheese chilli momo ...,112.0,4.4,0,soya cheese chilli momo
197426,Sikkim,Gangtok,2025-07-02,Mama's Kitchen,Gangtok,Momos,Kurkure momo fried ...,140.0,4.4,0,kurkure momo fried
197427,Sikkim,Gangtok,2025-03-25,Mama's Kitchen,Gangtok,Momos,Chilli cheese momo,126.0,4.4,0,chilli cheese momo
197428,Sikkim,Gangtok,2025-03-26,Mama's Kitchen,Gangtok,Momos,Veg Momos (8 Pc),85.0,4.4,0,veg momos


In [22]:
# Criamos um dicionário de prioridades
mapeamento = {
    'Sobremesas': [
        'ice cream', 'dessert', 'cake', 'kulfi', 'waffle', 'donut', 'brownie', 'sweet', 'falooda',
        'gulab', 'halwa', 'pastry', 'choco', 'pie', 'lava', 'sundae', 'brow-wow-nie',
        'mcflurry', 'rasmalai', 'shahi tukda', 'kaju katli', 'pudding', 'gelato', 'rasgulla',
        'kalakand', 'cookie', 'nutella', 'kitkat'
    ],
    'Bebidas': [
        'tea', 'coffee', 'juice', 'shake', 'drink', 'beverage', 'milk', 'soda', 'lassi', 'water',
        'mojito', 'cooler', 'coke', 'cola', 'pepsi', '7up', '7 up', 'mirinda', 'dew', 'thums up',
        'cappuccino', 'aam panna', 'sprite', 'coolberg', 'fizz', 'bottle', 'can ', 'fanta',
        'smoothie', 'latte', 'frappuccino', 'americano', 'cold brew', 'blonde', 'mocha',
        'lemonade', 'espresso', 'orange', 'b natural'
    ],
    'Pratos Principais': [
        'curry', 'gravy', 'paneer', 'chicken', 'mutton', 'dal', 'sabji', 'kadai', 'main course',
        'masala', 'kofta', 'fish', 'egg', 'veg', 'keema', 'pav bhaji', 'murg', 'makhni', 'chole',
        'lamb', 'aloo matar', 'do pyaza', 'mushroom', 'rajma', 'dum aloo'
    ],
    'Arroz & Biryani': ['biryani', 'rice', 'pulao', 'jeera', 'khichdi'],
    'South Indian': ['dosa', 'idli', 'vada', 'bath', 'pongal', 'upma', 'sambar', 'uttapam'],
    'Refeições & Combos': ['meal', 'combo', 'bucket', 'lunchbox', 'binge', 'squad', 'party', 'saving', 'duo', 'thali', 'box', 'make at home'],
    'Lanches': [
        'burger', 'sandwich', 'pizza', 'roll', 'wrap', 'fries', 'nuggets', 'momo', 'noodle',
        'pasta', 'chowmein', 'margherita', 'feast', 'mexican', 'crispy', 'zinger', 'whopper',
        'pepperoni', 'croissant', 'loaf', 'sourdough', 'baguette', 'cheesiken', 'delight'
    ],
    'Entradas & Petiscos': [
        'kebab', 'tikka', 'chaat', 'starter', 'appetizer', 'snack', 'manchurian', 'chilly',
        'wedges', 'shots', 'potato', 'garlic bread', 'stix', 'poppers', 'strips', 'popcorn',
        'wings', 'soup', 'chilli', 'samosa', 'kachori', 'bhalla'
    ],
    'Pães & Acompanhamentos': [
        'roti', 'naan', 'paratha', 'kulcha', 'phulka', 'chapati', 'raita', 'papad', 'chutney',
        'pickle', 'salad', 'dip', 'ketchup', 'mayo', 'onion', 'corn', 'cheese', 'curd', 'spice mix', 'sauce'
    ]
}
#def classificar_melhor(texto):
#    for categoria, palavras in mapeamento.items():
#        if any(p in texto for p in palavras):
#            return categoria
#    return 'Outros'
def classificar_final(texto):
    # Converte o nome do prato para minúsculo e remove espaços nas pontas
    t = str(texto).lower().strip()

    # Percorre o dicionário
    for categoria, palavras in mapeamento.items():
        # Verifica se alguma palavra-chave está CONTIDA no texto
        if any(p.lower() in t for p in palavras):
            return categoria

    return 'Outros'

# 3. Aplicando na coluna que você já limpou anteriormente
df['super_category'] = df['dish_clean'].apply(classificar_final)

# Aplicamos no nome que já limpamos
#df['super_category'] = df['dish_clean'].apply(classificar_melhor)

In [23]:
mapeamento = {
    'Sobremesas': ['ice cream', 'dessert', 'cake', 'kulfi', 'waffle', 'donut', 'brownie', 'sweet', 'falooda', 'gulab', 'halwa', 'pastry', 'choco', 'pie', 'lava', 'sundae', 'brow-wow-nie', 'mcflurry', 'rasmalai', 'shahi tukda', 'kaju katli', 'pudding', 'gelato', 'rasgulla', 'kalakand', 'cookie', 'nutella', 'kitkat','jalebi', 'laddu', 'sorbet', 'popsicle', 'fruit bowl', 'mini sticks','modak', 'peda', 'imarti', 'kheer', 'tiramisu', 'ice pops', 'almond rocks', 'jar'],
    'Bebidas': ['tea', 'coffee', 'juice', 'shake', 'drink', 'beverage', 'milk', 'soda', 'lassi', 'water', 'mojito', 'cooler', 'coke', 'cola', 'pepsi', '7up', '7 up', 'mirinda', 'dew', 'thums up', 'cappuccino', 'aam panna', 'sprite', 'coolberg', 'fizz', 'bottle', 'can ', 'fanta', 'smoothie', 'latte', 'frappuccino', 'americano', 'cold brew', 'blonde', 'mocha', 'lemonade', 'espresso', 'orange', 'b natural','frappe', 'blue lagoon', 'devils own', 'dark frappe', 'alphonso', 'green apple', 'mango ras','macchiato', 'go green'],
    'Pratos Principais': ['curry', 'gravy', 'paneer', 'chicken', 'mutton', 'dal', 'sabji', 'kadai', 'main course', 'masala', 'kofta', 'fish', 'egg', 'veg', 'keema', 'pav bhaji', 'murg', 'makhni', 'chole', 'lamb', 'aloo matar', 'do pyaza', 'mushroom', 'rajma', 'dum aloo', 'aloo gobi', 'aloo palak','aloo dum', 'aloo mattar', 'navratan korma', 'omelette','kadhi pakoda', 'ghee roast', 'pao bhaji', 'maggi', 'korma'],
    'Arroz & Biryani': ['biryani', 'rice', 'pulao', 'jeera', 'khichdi'],
    'South Indian': ['dosa', 'idli', 'vada', 'bath', 'pongal', 'upma', 'sambar', 'uttapam'],
    'Refeições & Combos': ['meal', 'combo', 'bucket', 'lunchbox', 'binge', 'squad', 'party', 'saving', 'duo', 'thali', 'box', 'make at home', 'hamper', 'basket', 'gift'],
    'Lanches': ['burger', 'sandwich', 'pizza', 'roll', 'wrap', 'fries', 'nuggets', 'momo', 'noodle', 'pasta', 'chowmein', 'margherita', 'feast', 'mexican', 'crispy', 'zinger', 'whopper', 'pepperoni', 'croissant', 'loaf', 'sourdough', 'baguette', 'cheesiken', 'delight', 'mcaloo', 'mcsaver','falafel', 'sausages', 'multipack', 'mini treats','spaghetti', 'aglio', 'fry', 'toasty'],
    'Entradas & Petiscos': ['kebab', 'tikka', 'chaat', 'starter', 'appetizer', 'snack', 'manchurian', 'chilly', 'wedges', 'shots', 'potato', 'garlic bread', 'stix', 'poppers', 'strips', 'popcorn', 'wings', 'soup', 'chilli', 'samosa', 'kachori', 'bhalla', 'murukku', 'puri', 'bhel','tikki', 'gobi 65', 'dhokla', 'chaap, poha','drums of heaven', 'tangdi', 'peri peri', 'chaap'],
    'Pães & Acompanhamentos': ['roti', 'naan', 'paratha', 'kulcha', 'phulka', 'chapati', 'raita', 'papad', 'chutney', 'pickle', 'salad', 'dip', 'ketchup', 'mayo', 'onion', 'corn', 'cheese', 'curd', 'spice mix', 'sauce', 'butter','seasoning', 'dahi, variety pack','dahi', 'marinara']
}

# 2. A função de classificação que resolve o problema do 7up e Murukku
def classificar_melhor(texto):
    # Força minúsculo, transforma em string e limpa espaços nas pontas
    # Isso resolve o problema de '7UP' vs '7up'
    t = str(texto).lower().strip()

    # Percorremos o dicionário
    for categoria, palavras in mapeamento.items():
        # Verificamos se alguma das palavras-chave está CONTIDA no texto
        # Isso resolve o 'butter murukku-200gm' porque ele acha a palavra 'butter' lá dentro
        if any(p.lower() in t for p in palavras):
            return categoria

    return 'Outros'

# 3. Aplicamos na coluna que você preferir (recomendo usar a 'Dish Name' original ou 'dish_clean')
df['super_category'] = df['Dish Name'].apply(classificar_melhor)

In [24]:
df

,State,City,Order Date,Restaurant Name,Location,Category,Dish Name,Price (INR),Rating,Rating Count,dish_clean,super_category
0,Karnataka,Bengaluru,2025-06-29,Anand Sweets & Savouries,Rajarajeshwari Nagar,Snack,Butter Murukku-200gm,133.9,4.0,0,butter murukku,Entradas & Petiscos
1,Karnataka,Bengaluru,2025-04-03,Srinidhi Sagar Deluxe,Kengeri,Recommended,Badam Milk,52.0,4.5,25,badam milk,Bebidas
2,Karnataka,Bengaluru,2025-01-15,Srinidhi Sagar Deluxe,Kengeri,Recommended,Chow Chow Bath,117.0,4.7,48,chow chow bath,South Indian
3,Karnataka,Bengaluru,2025-04-17,Srinidhi Sagar Deluxe,Kengeri,Recommended,Kesari Bath,65.0,4.6,65,kesari bath,South Indian
4,Karnataka,Bengaluru,2025-03-13,Srinidhi Sagar Deluxe,Kengeri,Recommended,Mix Raitha,130.0,4.0,0,mix raitha,Outros
...,...,...,...,...,...,...,...,...,...,...,...,...
197425,Sikkim,Gangtok,2025-01-25,Mama's Kitchen,Gangtok,Momos,Soya cheese chilli momo ...,112.0,4.4,0,soya cheese chilli momo,Lanches
197426,Sikkim,Gangtok,2025-07-02,Mama's Kitchen,Gangtok,Momos,Kurkure momo fried ...,140.0,4.4,0,kurkure momo fried,Lanches
197427,Sikkim,Gangtok,2025-03-25,Mama's Kitchen,Gangtok,Momos,Chilli cheese momo,126.0,4.4,0,chilli cheese momo,Lanches
197428,Sikkim,Gangtok,2025-03-26,Mama's Kitchen,Gangtok,Momos,Veg Momos (8 Pc),85.0,4.4,0,veg momos,Pratos Principais


In [25]:
# Contagem absoluta
contagem = df['super_category'].value_counts()
print(contagem)

super_category
Pratos Principais         91335
Sobremesas                30106
Bebidas                   22761
Lanches                   13750
Outros                    13140
Pães & Acompanhamentos     9396
Entradas & Petiscos        5947
Refeições & Combos         3977
Arroz & Biryani            3948
South Indian               3043
Name: count, dtype: int64


In [26]:
print(df[df['super_category'] == 'Outros']['Dish Name'].value_counts().head(30))
#print(df[df['super_category'] == 'Outros']['Dish Name'].value_counts().head(30))

Dish Name
Poha                                   24
Variety Pack                           20
Strawberry                             16
Aloo Tamatar                           14
Double Trouble                         14
Kaju Kalash                            14
Brown Bread                            14
Anjeer No Added Sugar 500ml            14
Kadhi Chawal                           13
Beetroot Pachadi                       13
Strawberry Icy Slush                   13
Fresh Lime                             13
Gujiya                                 13
Aloo Mutter                            13
Pineapple                              13
Angoori Jamun (12 Nos)                 13
Fruit Punch                            12
Mango                                  12
Red Velvet Slice                       12
Funwich Caramel Biscuit (90ml)         12
Kala Jamun                             12
Pork Chow                              12
Original Glazed Doughnut               12
The Muscle Multiplier   

In [27]:
df

,State,City,Order Date,Restaurant Name,Location,Category,Dish Name,Price (INR),Rating,Rating Count,dish_clean,super_category
0,Karnataka,Bengaluru,2025-06-29,Anand Sweets & Savouries,Rajarajeshwari Nagar,Snack,Butter Murukku-200gm,133.9,4.0,0,butter murukku,Entradas & Petiscos
1,Karnataka,Bengaluru,2025-04-03,Srinidhi Sagar Deluxe,Kengeri,Recommended,Badam Milk,52.0,4.5,25,badam milk,Bebidas
2,Karnataka,Bengaluru,2025-01-15,Srinidhi Sagar Deluxe,Kengeri,Recommended,Chow Chow Bath,117.0,4.7,48,chow chow bath,South Indian
3,Karnataka,Bengaluru,2025-04-17,Srinidhi Sagar Deluxe,Kengeri,Recommended,Kesari Bath,65.0,4.6,65,kesari bath,South Indian
4,Karnataka,Bengaluru,2025-03-13,Srinidhi Sagar Deluxe,Kengeri,Recommended,Mix Raitha,130.0,4.0,0,mix raitha,Outros
...,...,...,...,...,...,...,...,...,...,...,...,...
197425,Sikkim,Gangtok,2025-01-25,Mama's Kitchen,Gangtok,Momos,Soya cheese chilli momo ...,112.0,4.4,0,soya cheese chilli momo,Lanches
197426,Sikkim,Gangtok,2025-07-02,Mama's Kitchen,Gangtok,Momos,Kurkure momo fried ...,140.0,4.4,0,kurkure momo fried,Lanches
197427,Sikkim,Gangtok,2025-03-25,Mama's Kitchen,Gangtok,Momos,Chilli cheese momo,126.0,4.4,0,chilli cheese momo,Lanches
197428,Sikkim,Gangtok,2025-03-26,Mama's Kitchen,Gangtok,Momos,Veg Momos (8 Pc),85.0,4.4,0,veg momos,Pratos Principais


###até aqui

In [28]:
print(df['super_category'].nunique())

10


In [29]:
from google.colab import files

# Salva temporariamente no servidor do Colab
#df.to_csv('swiggy_sc.csv', sep=';', encoding='utf-8-sig', index=False)

# Dispara o download automático para o seu navegador
#files.download('swiggy_sc.csv')

In [30]:
df1 = df[~df['dish_clean'].astype(str).isin(['0', 'nan', ''])]

In [31]:
df1

,State,City,Order Date,Restaurant Name,Location,Category,Dish Name,Price (INR),Rating,Rating Count,dish_clean,super_category
0,Karnataka,Bengaluru,2025-06-29,Anand Sweets & Savouries,Rajarajeshwari Nagar,Snack,Butter Murukku-200gm,133.9,4.0,0,butter murukku,Entradas & Petiscos
1,Karnataka,Bengaluru,2025-04-03,Srinidhi Sagar Deluxe,Kengeri,Recommended,Badam Milk,52.0,4.5,25,badam milk,Bebidas
2,Karnataka,Bengaluru,2025-01-15,Srinidhi Sagar Deluxe,Kengeri,Recommended,Chow Chow Bath,117.0,4.7,48,chow chow bath,South Indian
3,Karnataka,Bengaluru,2025-04-17,Srinidhi Sagar Deluxe,Kengeri,Recommended,Kesari Bath,65.0,4.6,65,kesari bath,South Indian
4,Karnataka,Bengaluru,2025-03-13,Srinidhi Sagar Deluxe,Kengeri,Recommended,Mix Raitha,130.0,4.0,0,mix raitha,Outros
...,...,...,...,...,...,...,...,...,...,...,...,...
197425,Sikkim,Gangtok,2025-01-25,Mama's Kitchen,Gangtok,Momos,Soya cheese chilli momo ...,112.0,4.4,0,soya cheese chilli momo,Lanches
197426,Sikkim,Gangtok,2025-07-02,Mama's Kitchen,Gangtok,Momos,Kurkure momo fried ...,140.0,4.4,0,kurkure momo fried,Lanches
197427,Sikkim,Gangtok,2025-03-25,Mama's Kitchen,Gangtok,Momos,Chilli cheese momo,126.0,4.4,0,chilli cheese momo,Lanches
197428,Sikkim,Gangtok,2025-03-26,Mama's Kitchen,Gangtok,Momos,Veg Momos (8 Pc),85.0,4.4,0,veg momos,Pratos Principais


In [32]:
#from google.colab import files


# Solicita o upload do arquivo
print("Selecione o arquivo df_f.csv :")
uploaded = files.upload()

# Carrega o DataFrame
file_name = list(uploaded.keys())[0]
df_final2 = pd.read_csv(io.BytesIO(uploaded[file_name]))

print(f"\n✅ Arquivo '{file_name}' carregado com sucesso!")
print(f"Total de registros: {len(df_final2)}")

Selecione o arquivo df_f.csv :


Saving df_final2.csv to df_final2 (1).csv

✅ Arquivo 'df_final2 (1).csv' carregado com sucesso!
Total de registros: 197356


In [33]:
df_final2

,State,City,Order Date,Restaurant Name,Location,Category,Dish Name,Price,Rating,Rating Count,dish_clean,super_category
0,Karnataka,Bengaluru,29/06/2025,Anand Sweets & Savouries,Rajarajeshwari Nagar,Snack,Butter Murukku-200gm,133.9,4.0,0,butter murukku,Entradas & Petiscos
1,Karnataka,Bengaluru,03/04/2025,Srinidhi Sagar Deluxe,Kengeri,Recommended,Badam Milk,52.0,4.5,25,badam milk,Bebidas
2,Karnataka,Bengaluru,15/01/2025,Srinidhi Sagar Deluxe,Kengeri,Recommended,Chow Chow Bath,117.0,4.7,48,chow chow bath,South Indian
3,Karnataka,Bengaluru,17/04/2025,Srinidhi Sagar Deluxe,Kengeri,Recommended,Kesari Bath,65.0,4.6,65,kesari bath,South Indian
4,Karnataka,Bengaluru,13/03/2025,Srinidhi Sagar Deluxe,Kengeri,Recommended,Mix Raitha,130.0,4.0,0,mix raitha,Outros
...,...,...,...,...,...,...,...,...,...,...,...,...
197351,Sikkim,Gangtok,25/01/2025,Mama's Kitchen,Gangtok,Momos,Soya cheese chilli momo ...,112.0,4.4,0,soya cheese chilli momo,Lanches
197352,Sikkim,Gangtok,02/07/2025,Mama's Kitchen,Gangtok,Momos,Kurkure momo fried ...,140.0,4.4,0,kurkure momo fried,Lanches
197353,Sikkim,Gangtok,25/03/2025,Mama's Kitchen,Gangtok,Momos,Chilli cheese momo,126.0,4.4,0,chilli cheese momo,Lanches
197354,Sikkim,Gangtok,26/03/2025,Mama's Kitchen,Gangtok,Momos,Veg Momos (8 Pc),85.0,4.4,0,veg momos,Pratos Principais


###Modelo Star Schema

In [34]:
import pandas as pd

# --- PREPARAÇÃO ---
# Garante que a data é data e remove o que deu erro (as 166 linhas/erros)
df_final2['Order Date'] = pd.to_datetime(df_final2['Order Date'], format='%d/%m/%Y', errors='coerce')
df_final2 = df_final2.dropna(subset=['Order Date'])

# --- 1. DIM_RESTAURANT ---
dim_restaurant = df_final2[['Restaurant Name']].drop_duplicates().reset_index(drop=True)
dim_restaurant['restaurant_key'] = dim_restaurant.index + 1 # Começar em 1 é melhor prática

# --- 2. DIM_LOCATION ---
dim_location = df_final2[['State', 'City', 'Location']].drop_duplicates().reset_index(drop=True)
dim_location['location_key'] = dim_location.index + 1

# --- 3. DIM_DISH ---
dim_dish = df_final2[['Dish Name', 'dish_clean', 'Category', 'super_category']].drop_duplicates().reset_index(drop=True)
dim_dish['dish_key'] = dim_dish.index + 1

# --- 4. DIM_DATE (O ajuste fino) ---
# Pegamos os únicos, removemos nulos e garantimos que é datetime
unique_dates = df_final2['Order Date'].dropna().unique()
dim_date = pd.DataFrame({'full_date': unique_dates})
dim_date['full_date'] = pd.to_datetime(dim_date['full_date'])

# Agora o .dt funciona seguro
dim_date['date_key'] = dim_date['full_date'].dt.strftime('%Y%m%d').astype(int)
dim_date['day'] = dim_date['full_date'].dt.day
dim_date['month'] = dim_date['full_date'].dt.month
dim_date['year'] = dim_date['full_date'].dt.year

# --- 5. MAPEAMENTO (Criação da Fato) ---
# Usamos um DF temporário para não sujar o original
fact_orders = df_final2.merge(dim_restaurant, on='Restaurant Name') \
                       .merge(dim_location, on=['State', 'City', 'Location']) \
                       .merge(dim_dish, on=['Dish Name', 'dish_clean', 'Category', 'super_category'])

# Criar a chave de data na fato de forma segura
fact_orders['date_key'] = fact_orders['Order Date'].dt.strftime('%Y%m%d').astype(int)

# Seleção final das colunas da Fato
fact_orders = fact_orders[['date_key', 'restaurant_key', 'dish_key', 'location_key', 'Price', 'Rating', 'Rating Count']]

In [35]:
dim_restaurant

,Restaurant Name,restaurant_key
0,Anand Sweets & Savouries,1
1,Srinidhi Sagar Deluxe,2
2,Thalassery Restaurant,3
3,Appu Donne Biriyani Palace,4
4,Bismillah Taj Hotel,5
...,...,...
988,Valley Vista,989
989,Etho Metho,990
990,Roll Corner,991
991,Hot Stuff Fast Food,992


In [36]:
dim_date

,full_date,date_key,day,month,year
0,2025-06-29,20250629,29,6,2025
1,2025-04-03,20250403,3,4,2025
2,2025-01-15,20250115,15,1,2025
3,2025-04-17,20250417,17,4,2025
4,2025-03-13,20250313,13,3,2025
...,...,...,...,...,...
238,2025-03-07,20250307,7,3,2025
239,2025-03-21,20250321,21,3,2025
240,2025-03-02,20250302,2,3,2025
241,2025-03-27,20250327,27,3,2025


In [37]:
fact_orders

,date_key,restaurant_key,dish_key,location_key,Price,Rating,Rating Count
0,20250629,1,1,1,133.9,4.0,0
1,20250403,2,2,2,52.0,4.5,25
2,20250115,2,3,2,117.0,4.7,48
3,20250417,2,4,2,65.0,4.6,65
4,20250313,2,5,2,130.0,4.0,0
...,...,...,...,...,...,...,...
197351,20250125,993,82840,989,112.0,4.4,0
197352,20250702,993,82841,989,140.0,4.4,0
197353,20250325,993,82842,989,126.0,4.4,0
197354,20250326,993,82843,989,85.0,4.4,0


###Verif

In [38]:
original_rows = len(df_final2)
fact_rows = len(fact_orders)

if original_rows == fact_rows:
    print(f"✅ Sucesso! Volume preservado: {fact_rows} linhas.")
else:
    print(f"❌ Erro! Diferença de {original_rows - fact_rows} linhas. Verifique seus merges!")

✅ Sucesso! Volume preservado: 197356 linhas.


In [39]:
"""
import pandas as pd
from google.colab import files

# --- 1. CONFIGURAÇÃO DO DICIONÁRIO DE TABELAS ---
# Certifique-se de que esses nomes de DataFrames já foram criados no seu notebook
model_tables = {
     "fact_orders": fact_orders,
    "dim_dish": dim_dish,
    "dim_restaurant": dim_restaurant,
    "dim_location": dim_location,
    "dim_date": dim_date
}

# --- 2. EXPORTAÇÃO PARA PARQUET NO AMBIENTE VIRTUAL ---
print("🚀 Iniciando exportação para Parquet...")

exported_files = []
for table_name, df in model_tables.items():
    file_path = f"{table_name}.parquet"

    # Exporta sem o índice e com compressão Snappy (padrão de mercado)
    df.to_parquet(file_path, index=False, compression='snappy')

    exported_files.append(file_path)
    print(f"✅ Tabela '{table_name}' salva no Colab.")

print("\n--- Exportação Concluída ---\n")

# --- 3. DOWNLOAD PARA O COMPUTADOR LOCAL ---
print("📥 Iniciando downloads... Por favor, autorize os múltiplos downloads no seu navegador.")

for file in exported_files:
    files.download(file)

"""

'\nimport pandas as pd\nfrom google.colab import files\n\n# --- 1. CONFIGURAÇÃO DO DICIONÁRIO DE TABELAS ---\n# Certifique-se de que esses nomes de DataFrames já foram criados no seu notebook\nmodel_tables = {\n     "fact_orders": fact_orders,\n    "dim_dish": dim_dish,\n    "dim_restaurant": dim_restaurant,\n    "dim_location": dim_location,\n    "dim_date": dim_date\n}\n\n# --- 2. EXPORTAÇÃO PARA PARQUET NO AMBIENTE VIRTUAL ---\nprint("🚀 Iniciando exportação para Parquet...")\n\nexported_files = []\nfor table_name, df in model_tables.items():\n    file_path = f"{table_name}.parquet"\n    \n    # Exporta sem o índice e com compressão Snappy (padrão de mercado)\n    df.to_parquet(file_path, index=False, compression=\'snappy\')\n    \n    exported_files.append(file_path)\n    print(f"✅ Tabela \'{table_name}\' salva no Colab.")\n\nprint("\n--- Exportação Concluída ---\n")\n\n# --- 3. DOWNLOAD PARA O COMPUTADOR LOCAL ---\nprint("📥 Iniciando downloads... Por favor, autorize os múltiplos do

In [40]:
import pandas as pd
from google.colab import files

# --- 1. CONFIGURAÇÃO DO DICIONÁRIO DE TABELAS ---
model_tables = {
    "fact_orders": fact_orders,
    "dim_dish": dim_dish,
    "dim_restaurant": dim_restaurant,
    "dim_location": dim_location,
    "dim_date": dim_date
}

# --- 2. EXPORTAÇÃO PARA CSV NO AMBIENTE VIRTUAL ---
print("🚀 Iniciando exportação para CSV...")

exported_files = []
for table_name, df in model_tables.items():
    file_path = f"{table_name}.csv"

    # index=False: não salva a coluna de números do Pandas
    # sep=',': separador padrão por vírgula
    # encoding='utf-8-sig': garante que acentos fiquem corretos no Excel/Dadosfera
    df.to_csv(file_path, index=False, sep=',', encoding='utf-8-sig')

    exported_files.append(file_path)
    print(f"✅ Tabela '{table_name}' salva no Colab.")

print("\n--- Exportação Concluída ---\n")

# --- 3. DOWNLOAD PARA O COMPUTADOR LOCAL ---
print("📥 Iniciando downloads... Autorize os pop-ups no navegador.")

for file in exported_files:
    files.download(file)

🚀 Iniciando exportação para CSV...
✅ Tabela 'fact_orders' salva no Colab.
✅ Tabela 'dim_dish' salva no Colab.
✅ Tabela 'dim_restaurant' salva no Colab.
✅ Tabela 'dim_location' salva no Colab.
✅ Tabela 'dim_date' salva no Colab.

--- Exportação Concluída ---

📥 Iniciando downloads... Autorize os pop-ups no navegador.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>